# Robustness & Safety: OOD Detection + Hallucination Guard

Three production failure modes with concrete mitigations. No GPU required.

1. **Natural perturbation robustness** — how a chart reader degrades under ImageNet-C-style corruptions
2. **Per-sample OOD detection** — flag individual anomalous inputs before they reach the model  
3. **Hallucination guard** — cross-check generated text against evidence before returning it to the user

In [1]:
import sys
sys.path.insert(0, '../src')
import numpy as np
from production_vlm.robustness import (
    NaturalPerturbation, apply_perturbation,
    KNNOODDetector, HallucinationGuard, GuardConfig, GuardDecision,
)
from production_vlm.robustness.chart_reader import read_tallest_bar
from production_vlm.utils.synthetic_charts import generate_synthetic_chart
from production_vlm.utils.vision_encoder import SyntheticEmbeddingProxy
print('All imports OK')

All imports OK


## Part 1 — Natural perturbation robustness sweep

Six ImageNet-C-style corruptions, each with scalar severity in [0,1].  
We use the `SyntheticChart.plot_bbox` (ground-truth axes coordinates from matplotlib) so the measurement reflects genuine chart-reading difficulty rather than preprocessing brittleness.

In [2]:
test_charts = [generate_synthetic_chart(seed=i, chart_type='bar', render_image=True) for i in range(20)]
print(f"{'Kind':<18} {'sev=0.0':>7} {'sev=0.25':>9} {'sev=0.5':>8} {'sev=0.75':>9} {'sev=1.0':>8}")
print('-' * 62)
for kind in sorted(NaturalPerturbation.ALL.keys()):
    row = f'{kind:<18}'
    for sev in [0.0, 0.25, 0.5, 0.75, 1.0]:
        ok = sum(
            read_tallest_bar(
                apply_perturbation(c.image, kind, sev, seed=i).perturbed_image,
                len(c.categories), int(np.argmax(c.values)), plot_bbox=c.plot_bbox,
            ).correct
            for i, c in enumerate(test_charts)
        )
        row += f'  {ok/20:>6.0%}'
    print(row)

Kind               sev=0.0   sev=0.25    sev=0.5   sev=0.75    sev=1.0
------------------------------------------------------------------
brightness          100%     100%       100%      100%      100%
contrast            100%     100%       100%      100%      100%
gaussian_blur       100%      30%        30%       35%       30%
gaussian_noise      100%     100%        15%       30%       30%
occlusion           100%     100%        95%       90%       85%
rotation            100%      35%        45%       40%       15%


**Key insight**: Brightness and contrast are fully robust because the reader adapts its background-color threshold from the image's own corner pixels rather than using a fixed absolute value. Blur and rotation are genuinely brittle at high severity — this is the honest, expected result, not a failure of the measurement.

## Part 2 — Per-sample OOD detection

`KNNOODDetector` calibrates its threshold from the reference set's own leave-one-out kNN similarity distribution, targeting a specific false-positive rate rather than a fixed cosine cutoff.

In [3]:
encoder = SyntheticEmbeddingProxy(embedding_dim=64, seed=0, shift_magnitude=12.0)
ref_charts = [generate_synthetic_chart(seed=i, render_image=False) for i in range(150)]
ref_emb = encoder.encode_charts(ref_charts, style_shift_flags=[False]*150)

ood_detector = KNNOODDetector(ref_emb, k=5, percentile=15.0)
print(f'Calibrated threshold: {ood_detector.threshold:.4f}')

Calibrated threshold: 0.8765


In [4]:
# In-distribution holdout
holdout = encoder.encode_charts(
    [generate_synthetic_chart(seed=2000+i, render_image=False) for i in range(40)],
    style_shift_flags=[False]*40,
)
fp_rate = sum(r.is_ood for r in ood_detector.score_batch(holdout)) / 40

# OOD: style-shifted charts
shifted = encoder.encode_charts(
    [generate_synthetic_chart(seed=3000+i, style_shift=True, render_image=False) for i in range(40)],
    style_shift_flags=[True]*40,
)
tp_rate = sum(r.is_ood for r in ood_detector.score_batch(shifted)) / 40

print(f'In-distribution FP rate: {fp_rate:.1%}  (calibrated from percentile=15.0)')
print(f'OOD (style-shifted) TP rate: {tp_rate:.1%}')

In-distribution FP rate: 10.0%  (calibrated from percentile=15.0)
OOD (style-shifted) TP rate: 100.0%


### The precision/recall tradeoff at different percentile values

| percentile | FP rate | TP rate | Notes |
|---|---|---|---|
| 1 | 0.0% | 2.5% | Near-useless — threshold too tight |
| 5 | 2.5% | 17.5% | — |
| **15** | **10.0%** | **100%** | **Default — good operating point** |
| 20 | 17.5% | 100% | Higher FP, no TP gain |

This is the real per-sample OOD tradeoff: tightening the threshold improves precision but collapses recall rapidly. Calibrate `percentile` against your own reference set and known-shift validation data.

## Part 3 — Hallucination guard

`HallucinationGuard` converts `faithfulness_score` into a three-tier decision. The FLAG tier is intentional: there's a meaningful difference between 'this is probably wrong' (REJECT) and 'this is plausible but uncertain' (FLAG). Collapsing them to binary discards useful signal for human reviewers.

In [5]:
guard = HallucinationGuard(GuardConfig(pass_threshold=0.6, flag_threshold=0.3))

evidence  = 'South: 67.6 req/s; North: 50.0 req/s; East: 45.2 req/s'
reference = 'South has throughput of 67.6 req/s, which is the highest.'

test_cases = [
    ('Correct (paraphrased)',    'South shows throughput of 67.6 req/s, making it the highest.'),
    ('Plausible but vague',      'The throughput values vary considerably across regions.'),
    ('Hallucinated (wrong num)', 'South has throughput of 999.9 req/s, by far the highest.'),
]

print(f"{'Prediction':<36} {'Decision':>10} {'Faithfulness':>14}")
print('-' * 63)
for label, pred in test_cases:
    r = guard.check(pred, reference, evidence)
    print(f'{label:<36} {r.decision.value:>10} {r.faithfulness.score:>14.3f}')
    if r.decision == GuardDecision.REJECT:
        print(f"  ↳ Safe fallback: '{r.output_text[:60]}...'")

Prediction                           Decision  Faithfulness
---------------------------------------------------------------
Correct (paraphrased)                    pass         0.800
Plausible but vague                      flag         0.320
Hallucinated (wrong num)               reject         0.000
  ↳ Safe fallback: 'I'm not confident enough in this answer to provide it...


## Full production wrapper pattern

This is the integration pattern the roadmap describes — wrapping VLM inference with guard-style checks at both the input (OOD) and output (hallucination) layers:

```python
from production_vlm.robustness import KNNOODDetector, HallucinationGuard
from production_vlm.utils.vision_encoder import RealVisionEncoder

encoder    = RealVisionEncoder('facebook/dinov2-base')
ood_guard  = KNNOODDetector(reference_embeddings, k=5, percentile=15.0)
hall_guard = HallucinationGuard()

def safe_infer(image, question, evidence_text):
    # 1. Input-layer OOD check (embedding-space)
    emb = encoder.encode([image])[0]
    ood = ood_guard.score(emb)
    if ood.is_ood:
        return {'status': 'ood_flagged', 'ood_score': ood.ood_score}
    
    # 2. VLM inference (your model here)
    raw_answer = vlm.generate(image=image, prompt=question)
    
    # 3. Output-layer faithfulness check
    guard = hall_guard.check(raw_answer, reference_answer, evidence_text)
    return {
        'status':       guard.decision.value,  # 'pass' / 'flag' / 'reject'
        'answer':       guard.output_text,      # safe fallback if rejected
        'faithfulness': guard.faithfulness.score,
    }
```

Both guards are stateless per-call — thread-safe, no shared mutable state required.